In [ ]:
# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================
!pip install -q torch transformers pillow gradio gTTS

import os
import torch
from PIL import Image
from transformers import ViTImageProcessor, GPT2TokenizerFast, VisionEncoderDecoderModel
from gtts import gTTS
import gradio as gr

from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. LOAD MODEL FROM DRIVE
# ==========================================
MODEL_PATH = "/content/drive/My Drive/Cricket_Commentary_Project/final_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ Loading model from {MODEL_PATH}...")

try:
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH).to(device)
    tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_PATH)
    feature_extractor = ViTImageProcessor.from_pretrained(MODEL_PATH)
    print("✅ Model loaded successfully!")
except OSError:
    print("❌ Error: Model not found. Please run the Training Notebook first.")

# ==========================================
# 3. INFERENCE FUNCTIONS
# ==========================================

def text_to_audio(text, output_path="commentary.mp3"):
    """Converts text to speech using Google TTS."""
    tts = gTTS(text=text, lang='en', tld='co.uk') # 'co.uk' for a British cricket accent
    tts.save(output_path)
    return output_path

def generate_commentary(image):
    """Main pipeline: Image -> Text -> Audio"""
    if image is None: return "Please upload an image.", None

    # 1. Preprocess Image
    img = Image.fromarray(image).convert("RGB")
    pixel_values = feature_extractor(images=img, return_tensors="pt").pixel_values.to(device)

    # 2. Generate Text
    gen_ids = model.generate(
        pixel_values,
        max_length=50,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    caption = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    # Clean up punctuation
    caption = caption.split('.')[0] + "."

    # 3. Generate Audio
    audio_path = text_to_audio(caption)

    return caption, audio_path

# ==========================================
# 4. GRADIO INTERFACE
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🏏 AI Cricket Commentator")
    gr.Markdown("Upload a cricket shot image, and the AI will generate commentary and speak it out!")

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="numpy", label="Upload Cricket Shot")
            btn = gr.Button("🎙️ Generate Commentary", variant="primary")

        with gr.Column():
            text_output = gr.Textbox(label="Commentary", lines=2)
            audio_output = gr.Audio(label="Audio Commentary")

    btn.click(generate_commentary, inputs=img_input, outputs=[text_output, audio_output])

# Launch
app.launch(debug=True)